In [11]:
!pip install langchain langchain-groq

In [12]:
pip install -U langchain langchain-core langchain-groq

Note: you may need to restart the kernel to use updated packages.


In [13]:
import os
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq

# API key
os.environ["GROQ_API_KEY"] = "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"

# Model
llm = ChatGroq(model="llama-3.1-8b-instant")

# Resume
resume = """
Experience: 5 years Data Scientist
Skills: Python, Machine Learning, NLP, SQL
Tools: TensorFlow, Tableau
"""

# Prompt
prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
Extract:
- Skills
- Experience
- Tools

Resume:
{resume}

Return JSON format.
Do NOT assume anything not present.
"""
)

# Run
final_prompt = prompt.format(resume=resume)

response = llm.invoke(final_prompt)

print(response)

content='```json\n{\n  "Skills": ["Python", "Machine Learning", "NLP", "SQL"],\n  "Experience": "5 years Data Scientist",\n  "Tools": ["TensorFlow", "Tableau"]\n}\n```' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 47, 'prompt_tokens': 86, 'total_tokens': 133, 'completion_time': 0.136528638, 'completion_tokens_details': None, 'prompt_time': 0.008868708, 'prompt_tokens_details': None, 'queue_time': 0.089061953, 'total_time': 0.145397346}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_4387d3edbb', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019d910f-bcba-7b62-a3c4-f286011aaf5d-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 86, 'output_tokens': 47, 'total_tokens': 133}


In [14]:
# Install (run once in Jupyter)
# !pip install langchain langchain-core langchain-groq

import os
from langchain_core.prompts import PromptTemplate
from langchain_groq import ChatGroq

# API Key
os.environ["GROQ_API_KEY"] = "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"

# Model (updated working model)
llm = ChatGroq(model="llama-3.1-8b-instant")


# ✅ STEP 2: Inputs


job_description = """
We are hiring a Data Scientist.

Requirements:
- Python
- Machine Learning
- NLP
- SQL
- Data Visualization
- 3+ years experience
"""

resume = """
Experience: 5 years Data Scientist
Skills: Python, Machine Learning, NLP, SQL
Tools: TensorFlow, Tableau
"""


#STEP 3: Skill Extraction


extract_prompt = PromptTemplate(
    input_variables=["resume"],
    template="""
Extract:
- Skills
- Experience
- Tools

Resume:
{resume}

Return JSON format.
Do NOT assume anything not present.
"""
)

extract_input = extract_prompt.format(resume=resume)
extract_output = llm.invoke(extract_input)

print("=== Extracted Data ===")
print(extract_output.content)


# STEP 4: Matching Logic


match_prompt = PromptTemplate(
    input_variables=["resume_data", "job"],
    template="""
You are an AI recruiter.

Compare the candidate resume with the job description.

Resume:
{resume_data}

Job Description:
{job}

Tasks:
1. List matching skills
2. List missing skills

Rules:
- Do NOT assume skills
- Only use given data

Return output in JSON format:

{{
  "matching_skills": [],
  "missing_skills": []
}}
"""
)

match_input = match_prompt.format(
    resume_data=extract_output.content,
    job=job_description
)

match_output = llm.invoke(match_input)

print("\n=== Matching Result ===")
print(match_output.content)

=== Extracted Data ===
Here's the extracted information in JSON format:

```json
{
  "Skills": ["Python", "Machine Learning", "NLP", "SQL"],
  "Experience": ["5 years Data Scientist"],
  "Tools": ["TensorFlow", "Tableau"]
}
```

=== Matching Result ===
Based on the given job description and candidate resume, I have compared the two and determined the matching and missing skills.

Here's the output in JSON format:

```json
{
  "matching_skills": ["Python", "Machine Learning", "NLP", "SQL"],
  "missing_skills": ["Data Visualization"]
}
```

Explanation:

- The candidate has the required skills of Python, Machine Learning, NLP, and SQL, which are listed in the job description.
- However, the job description requires Data Visualization, which is not mentioned in the candidate's resume. Therefore, Data Visualization is listed as a missing skill.


In [15]:
score_prompt = PromptTemplate(
    input_variables=["match_data"],
    template="""
You are an AI recruiter.

Based on the matching result below, assign a score from 0 to 100.

Matching Result:
{match_data}

Rules:
- More matching skills = higher score
- More missing skills = lower score
- Be realistic

Return output in JSON format:

{{
  "score": 0
}}
"""
)

score_input = score_prompt.format(
    match_data=match_output.content
)

score_output = llm.invoke(score_input)

print("=== Score ===")
print(score_output.content)

=== Score ===
Based on the given matching result, I will assign a score from 0 to 100. 

Since the candidate has 4 matching skills and 1 missing skill, I will assign a score based on the following formula:

Score = (Number of matching skills / (Number of matching skills + Number of missing skills)) * 100

In this case, the score would be:

Score = (4 / (4 + 1)) * 100
Score = (4 / 5) * 100
Score = 80

However, I want to be realistic. Considering that the candidate is missing one required skill (Data Visualization), which is not a minor skill, I will adjust the score downward.

Adjusted Score = 80 - 10 (for the missing skill)
Adjusted Score = 70

Here is the JSON output:

```json
{
  "score": 70
}
```


In [16]:
from langchain_core.prompts import PromptTemplate

explain_prompt = PromptTemplate(
    input_variables=["score", "match_data"],
    template="""
You are an AI recruiter.

Explain why this candidate received the given score.

Score:
{score}

Matching Result:
{match_data}

Guidelines:
- Mention strengths (matching skills)
- Mention weaknesses (missing skills)
- Keep it clear and professional

Return explanation in 3–5 lines.
"""
)

In [17]:
explain_input = explain_prompt.format(
    score=score_output.content,
    match_data=match_output.content
)

explain_output = llm.invoke(explain_input)

print("=== Explanation ===")
print(explain_output.content)

=== Explanation ===
Based on the matching result, the candidate demonstrates strong skills in Python, Machine Learning, NLP, and SQL, showcasing their technical expertise. However, a notable weakness is the lack of Data Visualization skills, which is a critical requirement for the position. This gap may impact the candidate's overall performance in the role.


In [18]:
resume_strong = """
Experience: 5 years Data Scientist
Skills: Python, Machine Learning, NLP, SQL, Deep Learning
Tools: TensorFlow, Tableau
"""

resume_avg = """
Experience: 2 years Data Analyst
Skills: Python, SQL
Tools: Excel, Power BI
"""

resume_weak = """
Experience: Fresher
Skills: Basic Excel
Tools: MS Office
"""

In [19]:
def run_pipeline(resume):

    # Step 1: Extract
    extract_input = extract_prompt.format(resume=resume)
    extract_output = llm.invoke(extract_input)

    # Step 2: Match
    match_input = match_prompt.format(
        resume_data=extract_output.content,
        job=job_description
    )
    match_output = llm.invoke(match_input)

    # Step 3: Score
    score_input = score_prompt.format(
        match_data=match_output.content
    )
    score_output = llm.invoke(score_input)

    # Step 4: Explain
    explain_input = explain_prompt.format(
        score=score_output.content,
        match_data=match_output.content
    )
    explain_output = llm.invoke(explain_input)

    return {
        "Extracted": extract_output.content,
        "Match": match_output.content,
        "Score": score_output.content,
        "Explanation": explain_output.content
    }



In [20]:
resume_strong = """
Experience: 5 years Data Scientist
Skills: Python, Machine Learning, NLP, SQL, Deep Learning
Tools: TensorFlow, Tableau
"""

resume_avg = """
Experience: 2 years Data Analyst
Skills: Python, SQL
Tools: Excel, Power BI
"""

resume_weak = """
Experience: Fresher
Skills: Basic Excel
Tools: MS Office
"""

In [22]:
print("===== STRONG CANDIDATE =====")
strong_result = run_pipeline(resume_strong)
print(strong_result)

print("\n===== AVERAGE CANDIDATE =====")
avg_result = run_pipeline(resume_avg)
print(avg_result)

print("\n===== WEAK CANDIDATE =====")
weak_result = run_pipeline(resume_weak)
print(weak_result)

===== STRONG CANDIDATE =====
{'Extracted': '```json\n{\n  "Skills": ["Python", "Machine Learning", "NLP", "SQL", "Deep Learning"],\n  "Experience": "5 years Data Scientist",\n  "Tools": ["TensorFlow", "Tableau"]\n}\n```', 'Match': 'Based on the given data, here\'s the comparison between the candidate\'s resume and the job description:\n\n```json\n{\n  "matching_skills": ["Python", "Machine Learning", "NLP", "SQL"],\n  "missing_skills": ["Data Visualization"]\n}\n```\n\nExplanation:\n\n- The candidate has the following matching skills:\n  - Python (present in both resume and job description)\n  - Machine Learning (present in both resume and job description)\n  - NLP (present in both resume and job description)\n  - SQL (present in both resume and job description)\n- The candidate is missing the following skill:\n  - Data Visualization (present in job description, but not in the candidate\'s resume)\n  Note that the job description also requires 3+ years of experience, but the candidate 

In [23]:
pip install langsmith

Note: you may need to restart the kernel to use updated packages.


In [24]:
import os

os.environ["LANGCHAIN_TRACING_V2"] = "true"
os.environ["LANGCHAIN_API_KEY"] = "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"
os.environ["LANGCHAIN_PROJECT"] = "resume-screening"

In [25]:
run_pipeline(resume_strong)
run_pipeline(resume_avg)
run_pipeline(resume_weak)

{'Extracted': '```json\n{\n  "Skills": ["Basic Excel"],\n  "Experience": ["Fresher"],\n  "Tools": ["MS Office"]\n}\n```',
 'Match': 'Based on the provided resume and job description, I will compare the two and provide the output in JSON format.\n\n**Matching Skills:**\nThere are no matching skills between the resume and the job description.\n\n**Missing Skills:**\nThe following skills are missing from the resume:\n\n- Python\n- Machine Learning\n- NLP\n- SQL\n- Data Visualization\n\nHere\'s the output in JSON format:\n\n```json\n{\n  "matching_skills": [],\n  "missing_skills": ["Python", "Machine Learning", "NLP", "SQL", "Data Visualization"]\n}\n```\n\nNote: The candidate\'s experience is listed as "Fresher", which typically means 0-2 years of experience. The job description requires 3+ years of experience, so the candidate\'s experience is not sufficient for the position.',
 'Score': 'Based on the provided information, I would assign a score of 0 to the candidate. Here\'s why:\n\n- T